# 1.4 VC dimension

The largest set of points a class can label in every possible way: the right notion of size when the class is infinite.

Thresholds have infinitely many parameter values, yet they cannot realize every labeling of two ordered points. VC dimension measures this behavioral capacity rather than counting parameters.

Save a copy to Drive to edit.

In [ ]:
import itertools
import math
import numpy as np
import matplotlib.pyplot as plt


def all_binary_labelings(m):
    return list(itertools.product([0, 1], repeat=m))


def threshold_labelings(points):
    points = np.asarray(points, dtype=float)
    candidates = [-np.inf]
    candidates.extend(((points[:-1] + points[1:]) / 2).tolist())
    candidates.append(np.inf)
    labels = set()
    for t in candidates:
        pred = tuple((points > t).astype(int).tolist())
        labels.add(pred)
    return labels


def interval_labelings(points):
    points = np.asarray(points, dtype=float)
    labels = {tuple(np.zeros(len(points), dtype=int).tolist())}
    for left in range(len(points)):
        for right in range(left, len(points)):
            pred = np.zeros(len(points), dtype=int)
            pred[left:right + 1] = 1
            labels.add(tuple(pred.tolist()))
    return labels


def realized_labelings(class_name, points):
    if class_name == "threshold":
        return threshold_labelings(points)
    if class_name == "interval":
        return interval_labelings(points)
    raise ValueError(class_name)


def sauer_growth(m, d):
    total = 0
    for i in range(d + 1):
        total += math.comb(m, i)
    return total


def shatters(class_name, points):
    return len(realized_labelings(class_name, points)) == 2 ** len(points)

## The concept, built once (D1)

A class shatters $m$ points if it realizes all $2^m$ binary labelings. For thresholds on the line, one point is shattered, but two ordered points cannot realize the pattern $(1,0)$.

In [ ]:
one_point = np.array([2.0])
two_points = np.array([1.0, 3.0])
labels_one = realized_labelings("threshold", one_point)
labels_two = realized_labelings("threshold", two_points)
print("one point labelings:", labels_one)
print("two point labelings:", labels_two)

The lesson's exact capacity facts are: thresholds shatter $1$ point but not $2$; intervals shatter $2$ but fail on $3$ because $(1,0,1)$ is unreachable; Sauer's lemma gives $16$ labelings for $d=2,m=5$.

In [ ]:
assert len(labels_one) == 2
assert len(labels_two) == 3
assert (1, 0) not in labels_two
assert shatters("interval", np.array([1.0, 2.0]))
assert not shatters("interval", np.array([1.0, 2.0, 3.0]))
assert (1, 0, 1) not in realized_labelings("interval", np.array([1.0, 2.0, 3.0]))
assert sauer_growth(5, 2) == 16
print("VC hand checks passed")

## The dataset ladder

D1 and D2 test threshold shattering directly, D3 tests interval shattering and failure, D4 records theory capacities for half-planes and rectangles, and D5 evaluates Sauer growth for larger samples.

In [ ]:
rungs = [
    {"name": "D1 threshold on 1 point", "m": 1, "realized": len(realized_labelings("threshold", np.array([1.0]))), "all": 2},
    {"name": "D2 threshold on 2 points", "m": 2, "realized": len(realized_labelings("threshold", np.array([1.0, 2.0]))), "all": 4},
    {"name": "D3 intervals on 3 points", "m": 3, "realized": len(realized_labelings("interval", np.array([1.0, 2.0, 3.0]))), "all": 8},
    {"name": "D4 half-plane and rectangle theory", "m": 4, "realized": 16, "all": 16, "vc_notes": "half-planes d=3, rectangles d=4"},
    {"name": "D5 Sauer d=2 growth", "m": 5, "realized": sauer_growth(5, 2), "all": 32},
]
for rung in rungs:
    print(rung)
print("D5 larger growth examples")
for m in [5, 6, 8, 10]:
    print(m, sauer_growth(m, 2), 2 ** m)

In [ ]:
print("rung | realized labelings | arbitrary labelings | ratio")
for rung in rungs:
    ratio = rung["realized"] / rung["all"]
    print(f"{rung['name']}: {rung['realized']} | {rung['all']} | {ratio:.3f}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(17, 6))
examples = [
    ("threshold", np.array([1.0])),
    ("threshold", np.array([1.0, 2.0])),
    ("interval", np.array([1.0, 2.0, 3.0])),
    ("theory", np.array([0.0, 1.0, 2.0, 3.0])),
    ("sauer", np.array([1.0, 2.0, 3.0, 4.0, 5.0])),
]
for ax, rung, example in zip(axes[0], rungs, examples):
    class_name, pts = example
    ax.scatter(pts, np.zeros_like(pts), s=50)
    ax.set_title(f"{rung['name']}
{rung['realized']}/{rung['all']} labelings")
    ax.set_yticks([])
    ax.grid(True, alpha=0.3)
ms = np.arange(1, 11)
growth = np.array([min(sauer_growth(int(m), 2), 2 ** int(m)) for m in ms])
all_labels = np.array([2 ** int(m) for m in ms])
axes[1, 0].plot(ms, growth, marker="o", label="VC d=2 Sauer")
axes[1, 0].plot(ms, all_labels, marker="o", label="2^m arbitrary")
axes[1, 0].set_yscale("log")
axes[1, 0].set_xlabel("m")
axes[1, 0].set_ylabel("labelings")
axes[1, 0].set_title("Payoff curve: growth function")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()

## Pitfall on the hardest rung

Pitfall: equating VC dimension with parameter count. The fix is to test realized labelings or use a valid growth-function theorem, not count knobs.

In [ ]:
m = 5
wrong_parameter_count_guess = 2
wrong_labelings = m ** wrong_parameter_count_guess
fixed_labelings = sauer_growth(m, 2)
arbitrary = 2 ** m
print(f"wrong parameter-count guess m^2={wrong_labelings}")
print(f"fixed Sauer count={fixed_labelings}")
print(f"arbitrary labelings={arbitrary}")
print("the behavior test is the certificate, not the parameter count")

## Evaluate it + Practice

- Metric: growth function and realized labelings.
- No-skill baseline: compare only the number of parameters.
- Cheap sanity check: verify a class shatters m points only when realized labelings equals 2^m.
- Ablation: replace realized_labelings with parameter counts and D2 gives the wrong lesson.
- Failure signals: a missing labeling, degeneracy, or a growth curve catching 2^m for too long.

### Practice prompts

- List the four interval labelings on two ordered points.

- Find the first missing threshold labeling on three ordered points.

- Compute Sauer growth for d=3 and m=10.